## Configuration

In [ ]:
import sys
sys.path.append('../src')

## Dataset

In [ ]:
DATA = '../.data/train_neutral_0s.npz' # single Frame dataset

import numpy as np
data = np.load(DATA)
x = np.nan_to_num(data['x'])
y = (data['y'].mean(axis = 1) > .5).astype(float)
x.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_eval, y_train, y_eval = train_test_split(
    x, y, test_size=0.5
)
(x_train.shape, y_train.shape), (x_eval.shape, y_eval.shape)

## Multi Layer Perceptron (Neural Network)

### Hyperparameter selection

f1 scoring function for parameter set
 * Builds MLP with total number of neurons devided into $n$ layers
 * Fits model on training dataset
 * Compute macro averaged f1 Score

In [ ]:
from models import MultiLabelMLP
from sklearn.metrics import f1_score
def score(n_neurons, n_layers):
    neurons_per_layer = n_neurons // n_layers
    hidden_layers = (neurons_per_layer,) * n_layers
    model = MultiLabelMLP(hidden_layers)
    model.fit(x_train, y_train, epochs=30, batch_size=1000)
    y_pred = model.predict(x_eval)
    return f1_score(y_eval, y_pred, average='macro')

Search strategies Grid and Random search

In [ ]:
def grid_search(neurons, layers):
    result = dict()
    for n in neurons:
        for l in layers:
            result[(n, l)] = score(n, l)
    return result

def random_search(neurons_min, neurons_max, layers_min, layers_max, n_samples):
    result = dict()
    for _ in range(n_samples):
        n = np.random.randint(neurons_min, neurons_max+1)
        l = np.random.randint(layers_min, layers_max+1)
        result[(n, l)] = score(n, l)
    return result

# Visualization
import matplotlib.pyplot as plt
def plot_search_results(results, title=''):
    p = np.array(tuple(results.keys()))
    a = np.array(tuple(results.values()))
    plt.scatter(*p.T, vmin=0, vmax=1, c=a)
    plt.xlabel('Total number of neurons')
    plt.ylabel('Number of layers')
    plt.colorbar(label='f1-score')
    plt.title(title)
    plt.show()

Perform random search

In [ ]:
rs = random_search(1, 5000, 1, 10, 50)
plot_search_results(rs, 'Random Search for MLP architecture')

Appropriate MLP acritecture requires
 * $n_{layers} \geq 3$ 
 * $n_{neurons} \geq 2000$

There is no significant change for increasing number of layers.
Thus, choose $n_{layers} = 4$

In [ ]:
neurons = (100, 200, 400, 800, 1500, 2500, 4000, 6000)
s = np.array([
    score(n, 4)
    for n in neurons
])
plt.scatter(neurons, s)
plt.plot(neurons, s)
plt.xlabel('Total number of neurons')
plt.ylabel('f1-score')
plt.ylim(0,1)
plt.title('MLP with 4 hidden layers')
plt.show()


Keep number of neurons small while having good f1 score.

=> Choose $n_{neurons} = 2400$

=> MLP with 4 layers a 600 neurons

In [ ]:
DATA = '../.data/train_neutral_%is.npz' # single Frame dataset

import numpy as np
from sklearn.model_selection import train_test_split

for d in (0,1,2,4,6,8):
    data = np.load(DATA % d)
    x = np.nan_to_num(data['x'])
    y = (data['y'].mean(axis = 1) > .5).astype(float)
    x_train, x_eval, y_train, y_eval = train_test_split(
        x, y, test_size=0.5
    )
    neurons = (100, 200, 400, 800, 1500, 2500, 4000, 6000)
    s = np.array([
        score(n, 4)
        for n in neurons
    ])
    plt.scatter(neurons, s)
    plt.plot(neurons, s, label=f"{d}s")

plt.xlabel('Total number of neurons')
plt.ylabel('f1-score')
plt.legend()
plt.ylim(0,1)
plt.title('MLP with 4 hidden layers')
plt.show()

    